# Traffic Signal AI — Colab Backend (dynamic pressure controller)

This version keeps the existing YOLOv8 + multi-lane pipeline but makes the signal controller visibly dynamic.
- A new decision is computed for every video segment.
- The lane with the highest current pressure is GREEN.
- Green duration changes with the winning pressure share.
- The annotated output displays CURRENT GREEN, pressure, pressure share, and vehicle count.
- `segment_decisions` contains the complete decision timeline.
- ESP32 sync is still performed after every segment.
- Segment length defaults to 2 seconds for faster visible changes.

Note: a signal lane changes only when the pressure ranking changes; if the same lane remains highest, its displayed pressure/green time can still change.


In [1]:
# CELL 1 — Install dependencies
!pip install -q ultralytics fastapi uvicorn pyngrok cloudinary requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.0 MB/s eta 0:00:00


In [2]:
# CELL 2 — Credentials & config
# Paste your own values below (get a fresh ngrok token from
# https://dashboard.ngrok.com/get-started/your-authtoken, your ngrok API key
# from https://dashboard.ngrok.com/api-keys, and your Cloudinary values from
# https://console.cloudinary.com/settings/api-keys).
# NOTE: if these are the same values from your original notebook, rotate them first —
# they were previously exposed in plain text.

import os

os.environ["CLOUDINARY_CLOUD_NAME"] = "hepjtv2u"
os.environ["CLOUDINARY_API_KEY"] = "367286211383646"
os.environ["CLOUDINARY_API_SECRET"] = "REPLACE_ME"
os.environ["NGROK_AUTHTOKEN"] = "REPLACE_ME"
os.environ["NGROK_API_KEY"] = "3Hxd5nR3Fs5meHIkNn3dnKDvBGV_v5NzFQNrg6uHw6jtp45n"

# URL of your deployed BACKEND (e.g. Render) -- NOT this Colab's own ngrok
# tunnel. Used so THIS Colab server can push each segment's signal decision
# to /esp32/sync as it's computed, so the physical ESP32 updates
# continuously while a long video is processed.
# ⚠️ Double-check this: it currently points at an ngrok-free.dev URL, which
# looks like Colab's OWN tunnel address, not your deployed backend. If so,
# Colab would be pushing decisions to itself instead of to your real
# backend/ESP32 relay -- replace with your actual Render (or similar) URL.
os.environ["BACKEND_URL"] = "https://YOUR_BACKEND_URL"

# How many seconds of *video* time each segment covers before a new
# decision is computed and pushed to the ESP32.
os.environ["SEGMENT_SECONDS"] = "5"

# How many seconds of video the very FIRST segment covers (short, so
# the first decision reaches SSE/ESP32 fast) before SEGMENT_SECONDS
# takes over for every segment after that.
os.environ["FIRST_SEGMENT_SECONDS"] = "1"
# Inference image size for the one-frame-per-lane INSTANT first decision
# (batched single YOLO call, runs before segment 0). Smaller = faster.
os.environ["FIRST_DECISION_IMGSZ"] = "384"

# ── Signal controller thresholds ──────────────────────────────────────
# Floor: how long a lane must hold green before it's even eligible to be
# swapped out early for clearing (prevents flip-flopping on one quiet segment).
os.environ["MIN_GREEN_SECONDS"] = "15"
# Ceiling: max continuous seconds the busiest lane can hold green before
# being forced to rotate to the next busiest OTHER lane (safety cap for
# traffic that never clears).
os.environ["MAX_GREEN_SECONDS"] = "60"
# Vehicle count at/below which the current green lane counts as "cleared"
# (traffic effectively ended) -- triggers an early switch instead of
# waiting out the full MAX_GREEN_SECONDS on an empty road.
os.environ["CLEAR_THRESHOLD_VEHICLES"] = "2"
# Max seconds a lane can wait before a small-but-real traffic count forces
# it to get an immediate turn, regardless of what's currently green.
os.environ["MAX_WAIT_SECONDS"] = "90"
# "Minimal traffic" range that qualifies a waiting lane for starvation relief.
os.environ["STARVATION_MIN_VEHICLES"] = "1"
os.environ["STARVATION_MAX_VEHICLES"] = "10"

placeholders = {k: v for k, v in os.environ.items() if v.startswith("PASTE_YOUR_")}
if placeholders:
    print("⚠️ Still using placeholder values for:", ", ".join(placeholders))
    print("Edit this cell and re-run before continuing.")
else:
    print("✓ Cloudinary credentials loaded")
    print("✓ ngrok token + API key loaded")
    print("✓ BACKEND_URL:", os.environ["BACKEND_URL"])
    print("✓ Controller: min_green=%ss max_green=%ss clear<=%s max_wait=%ss starvation=%s-%s vehicles" % (
        os.environ["MIN_GREEN_SECONDS"], os.environ["MAX_GREEN_SECONDS"], os.environ["CLEAR_THRESHOLD_VEHICLES"],
        os.environ["MAX_WAIT_SECONDS"], os.environ["STARVATION_MIN_VEHICLES"], os.environ["STARVATION_MAX_VEHICLES"]))

✓ Cloudinary credentials loaded
✓ ngrok token + API key loaded
✓ BACKEND_URL: https://YOUR_BACKEND_URL


In [ ]:
# CELL 3 — Write app.py (FastAPI + YOLOv8 + GREEDY/CLEAR-SWITCH/ROTATION/STARVATION-RELIEF CONTROLLER)
#
# SPEED: three separate latency cuts stack together for the FIRST decision:
#   1. Model warm-up at server boot (pays the "first inference is slow"
#      cost once, at startup, not on the user's request).
#   2. Lane videos download in PARALLEL (thread pool) instead of one after
#      another.
#   3. An "instant first decision": one frame per lane, run through a
#      SINGLE BATCHED YOLO call (not 4 separate calls) at a smaller
#      inference size, pushed immediately -- before the normal
#      multi-frame segment loop even starts. The real multi-frame segment
#      (FIRST_SEGMENT_SECONDS, default 1s) follows right after and
#      refines it.
#
# LATENCY FIX (segment loop): process_multi_lane_videos() used to read+
# detect a FULL SEGMENT_SECONDS worth of frames for lane 1, then lane 2,
# then lane 3, then lane 4, and only THEN publish a decision -- on CPU
# this made the first update take much longer than the segment length
# implied. The first segment after the instant pick above is now a short
# FIRST_SEGMENT_SECONDS window; every segment after that reverts to the
# normal, steadier SEGMENT_SECONDS.
#
# DIAGNOSTICS: each lane now prints video-opened/fps/frame-count/duration
# at load time (so a decode failure or 0-frame video is never silently
# swallowed into "0 vehicles"), and each segment prints raw YOLO detections
# (any class) separately from vehicle-class-filtered counts, so a "0
# vehicles" result can be told apart from "YOLO sees nothing" vs "YOLO sees
# things but none are cars/bikes/buses/trucks".
#
# No proportional-share math. Green goes DIRECTLY to whichever lane has
# the most traffic and STAYS there until one of, in priority order:
#   1) STARVATION RELIEF: a red lane waited MAX_WAIT_SECONDS with a small
#      but real vehicle count (STARVATION_MIN..MAX_VEHICLES) -> immediate
#      turn, overrides everything else.
#   2) TRAFFIC CLEARED: after at least MIN_GREEN_SECONDS, if the current
#      lane's count has dropped to <= CLEAR_THRESHOLD_VEHICLES AND another
#      lane has traffic waiting -> switches early instead of holding the
#      full max-green timer on a road that's basically empty.
#   3) MAX GREEN EXCEEDED: safety cap (MAX_GREEN_SECONDS) if the lane never
#      clears -> forced rotation to the busiest OTHER lane.
# Multi-lane mode ({"videos": {...}}) and legacy single-video quadrant mode
# ({"video_url": ...}) both supported. Live decisions pushed to
# {BACKEND_URL}/esp32/sync every SEGMENT_SECONDS.
app_code = r'''
import os, uuid, requests, time, torch, concurrent.futures
import cv2
import numpy as np
import cloudinary
import cloudinary.uploader
from ultralytics import YOLO
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

cloudinary.config(
    cloud_name=os.environ["CLOUDINARY_CLOUD_NAME"],
    api_key=os.environ["CLOUDINARY_API_KEY"],
    api_secret=os.environ["CLOUDINARY_API_SECRET"],
)

BACKEND_URL = os.environ.get("BACKEND_URL", "").rstrip("/")
SEGMENT_SECONDS = float(os.environ.get("SEGMENT_SECONDS", "5"))
# How long the VERY FIRST segment is, in seconds of source video. Every
# segment after the first uses the normal SEGMENT_SECONDS. This exists
# purely to get a first signal decision onto the SSE stream / ESP32 fast --
# waiting a full SEGMENT_SECONDS x 4-lanes on CPU before the frontend sees
# anything makes the UI look stuck. A short first segment still gives the
# controller a real (if noisier) sample to make an initial pick from.
FIRST_SEGMENT_SECONDS = float(os.environ.get("FIRST_SEGMENT_SECONDS", "1"))
# Inference image size used ONLY for the single-frame "instant first
# decision" batched YOLO call below (smaller = faster but slightly less
# accurate on small/far vehicles). Normal segments keep the model's
# default size for full accuracy; this one frame is deliberately traded
# for speed since it's a provisional pick that gets refined seconds later.
FIRST_DECISION_IMGSZ = int(os.environ.get("FIRST_DECISION_IMGSZ", "384"))

# ── Signal controller config ──────────────────────────────────────────
# MAX_GREEN_SECONDS: how long the busiest lane is allowed to hold green
#   before it is FORCED to rotate to whichever OTHER lane is currently
#   busiest, even if the original lane is still the single busiest lane
#   overall. This is what stops one heavy lane from hogging green forever.
# MAX_WAIT_SECONDS: if a red lane has been waiting this long, and its
#   traffic is small-but-real (between STARVATION_MIN_VEHICLES and
#   STARVATION_MAX_VEHICLES), it gets an immediate turn -- overriding
#   whatever is currently green, regardless of that lane's max-green timer.
#   This guarantees a lightly-loaded lane never waits forever behind a
#   perpetually busy one.
MAX_GREEN_SECONDS = float(os.environ.get("MAX_GREEN_SECONDS", "60"))
MAX_WAIT_SECONDS = float(os.environ.get("MAX_WAIT_SECONDS", "90"))
STARVATION_MIN_VEHICLES = int(os.environ.get("STARVATION_MIN_VEHICLES", "1"))
STARVATION_MAX_VEHICLES = int(os.environ.get("STARVATION_MAX_VEHICLES", "10"))
# MIN_GREEN_SECONDS: floor on how long a lane holds green before it's even
# eligible to be swapped out early for clearing -- prevents flip-flopping
# on a single quiet segment.
# CLEAR_THRESHOLD_VEHICLES: vehicle count at/below which the current green
# lane counts as "cleared" (traffic effectively ended), triggering an
# early switch to a lane that still has traffic -- instead of waiting out
# the full MAX_GREEN_SECONDS on a road that's basically empty.
MIN_GREEN_SECONDS = float(os.environ.get("MIN_GREEN_SECONDS", "15"))
CLEAR_THRESHOLD_VEHICLES = int(os.environ.get("CLEAR_THRESHOLD_VEHICLES", "2"))

# Enable GPU
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Loading YOLOv8n on device: {device}...")
model = YOLO('yolov8n.pt')
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

# Warm up the model once at startup (not per-request). A model's FIRST
# inference call is typically much slower than every call after it --
# CUDA kernels get compiled/cached, thread pools spin up, etc. Paying that
# cost here, once, at server boot means the actual first-request inference
# (the one the user is waiting on) runs at full speed instead of eating a
# multi-second one-time tax.
try:
    _t_warm = time.time()
    model.predict(np.zeros((640, 640, 3), dtype=np.uint8), device=device, verbose=False)
    print(f"✓ Model warmed up in {round(time.time() - _t_warm, 2)}s")
except Exception as _e:
    print(f"[warmup] skipped (non-fatal): {_e}")

app = FastAPI(title="Traffic AI Colab Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class VideoRequest(BaseModel):
    video_url: Optional[str] = None            # legacy mode: single video, quadrant-split
    videos: Optional[dict] = None               # new mode: {"North": url, "South": url, "East": url, "West": url}
    backend_url: Optional[str] = None           # per-request override of BACKEND_URL
    segment_seconds: Optional[float] = None     # per-request override of SEGMENT_SECONDS
    session_id: Optional[str] = None            # backend session id, for live SSE push to the frontend

ALL_LANES = ["North", "South", "East", "West"]

class SignalController:
    """
    Stateful greedy signal controller with 3 layered rules, checked in
    this priority order every segment:

      1. STARVATION RELIEF (highest priority): a red lane has waited
         >= MAX_WAIT_SECONDS and has a small-but-real vehicle count
         (STARVATION_MIN..MAX_VEHICLES). It gets green immediately,
         overriding everything else. Guarantees no lane waits forever.

      2. TRAFFIC CLEARED (efficiency): the current green lane has held
         green for at least MIN_GREEN_SECONDS (a floor, so it doesn't
         flip-flop on a single quiet segment) AND its count has dropped
         to <= CLEAR_THRESHOLD_VEHICLES (traffic has "ended", not
         necessarily to literal zero) AND at least one other lane has
         traffic waiting. Green moves early to the busiest other lane
         instead of sitting on a road that's basically empty.

      3. MAX GREEN EXCEEDED (safety cap): if neither of the above fired
         and the current lane has held green for MAX_GREEN_SECONDS, it
         is force-rotated to the busiest OTHER lane -- this is the
         backstop for a lane whose traffic never fully clears.

      Otherwise: the current green lane simply continues.
    """
    def __init__(self, lanes=None,
                 min_green_seconds=MIN_GREEN_SECONDS, max_green_seconds=MAX_GREEN_SECONDS,
                 max_wait_seconds=MAX_WAIT_SECONDS,
                 clear_threshold_vehicles=CLEAR_THRESHOLD_VEHICLES,
                 starvation_min=STARVATION_MIN_VEHICLES, starvation_max=STARVATION_MAX_VEHICLES):
        self.lanes = lanes or list(ALL_LANES)
        self.min_green_seconds = min_green_seconds
        self.max_green_seconds = max_green_seconds
        self.max_wait_seconds = max_wait_seconds
        self.clear_threshold_vehicles = clear_threshold_vehicles
        self.starvation_min = starvation_min
        self.starvation_max = starvation_max
        self.current_green = None
        self.green_elapsed = 0.0
        self.waiting_time = {lane: 0.0 for lane in self.lanes}

    def _switch_to(self, new_lane):
        self.current_green = new_lane
        self.green_elapsed = 0.0
        self.waiting_time[new_lane] = 0.0

    def decide(self, counts, segment_seconds):
        counts = {lane: max(0, int(counts.get(lane, 0))) for lane in self.lanes}
        reason = "initial"

        if self.current_green is None:
            candidates = [l for l in self.lanes if counts[l] > 0]
            self.current_green = max(candidates, key=lambda l: counts[l]) if candidates else self.lanes[0]
            self.green_elapsed = 0.0
        else:
            self.green_elapsed += segment_seconds
            for lane in self.lanes:
                if lane != self.current_green:
                    self.waiting_time[lane] += segment_seconds

            others = [l for l in self.lanes if l != self.current_green]
            busy_others = [l for l in others if counts[l] > 0]

            # 1. Starvation relief -- checked first, overrides everything.
            starved = [
                l for l in others
                if self.waiting_time[l] >= self.max_wait_seconds
                and self.starvation_min <= counts[l] <= self.starvation_max
            ]
            if starved:
                new_green = max(starved, key=lambda l: self.waiting_time[l])
                self._switch_to(new_green)
                reason = "starvation_relief"

            # 2. Traffic cleared -- only eligible after min green time, and
            #    only if someone else actually needs the road.
            elif (self.green_elapsed >= self.min_green_seconds
                  and counts[self.current_green] <= self.clear_threshold_vehicles
                  and busy_others):
                new_green = max(busy_others, key=lambda l: counts[l])
                self._switch_to(new_green)
                reason = "traffic_cleared"

            # 3. Max green exceeded -- safety cap for traffic that never clears.
            elif self.green_elapsed >= self.max_green_seconds:
                if busy_others:
                    new_green = max(busy_others, key=lambda l: counts[l])
                    self._switch_to(new_green)
                    reason = "max_green_exceeded"
                else:
                    reason = "max_green_exceeded_no_other_traffic"

            else:
                reason = "continue_green"

        decisions = {}
        for lane in self.lanes:
            decisions[lane] = {
                "signal": "GREEN" if lane == self.current_green else "RED",
                "vehicle_count": counts[lane],
                "waiting_time_seconds": 0.0 if lane == self.current_green else round(self.waiting_time[lane], 1),
            }
        decisions["_controller"] = {
            "selected_lane": self.current_green,
            "green_elapsed_seconds": round(self.green_elapsed, 1),
            "reason": reason,
            "min_green_seconds": self.min_green_seconds,
            "max_green_seconds": self.max_green_seconds,
            "max_wait_seconds": self.max_wait_seconds,
            "clear_threshold_vehicles": self.clear_threshold_vehicles,
        }
        return decisions
def divide_frame_into_lanes(frame):
    h, w = frame.shape[:2]
    return {
        "North": (0, 0, w // 2, h // 2),
        "South": (w // 2, 0, w, h // 2),
        "East":  (0, h // 2, w // 2, h),
        "West":  (w // 2, h // 2, w, h),
    }

def count_vehicles_in_lane(detections, lane_box):
    x1l, y1l, x2l, y2l = lane_box
    count = 0
    for det in detections:
        if int(det[5]) not in VEHICLE_CLASSES:
            continue
        cx, cy = (det[0] + det[2]) / 2, (det[1] + det[3]) / 2
        if x1l <= cx <= x2l and y1l <= cy <= y2l:
            count += 1
    return count

def draw_signal_overlay(frame, lanes, decisions):
    """Large, high-contrast overlay showing the CURRENT controller decision."""
    h, w = frame.shape[:2]
    colors = {"GREEN": (0, 220, 0), "RED": (0, 0, 255)}
    controller = decisions.get("_controller", {})
    selected = controller.get("selected_lane", "NONE")
    elapsed = controller.get("green_elapsed_seconds", 0)
    reason = controller.get("reason", "")

    banner = f"GREEN: {selected}  |  held {elapsed:.0f}s  |  {reason}"
    cv2.rectangle(frame, (0, 0), (w, 48), (30, 30, 30), -1)
    cv2.putText(frame, banner, (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.62,
                (0, 255, 0) if selected != "NONE" else (255, 255, 255), 2)

    cv2.line(frame, (w // 2, 48), (w // 2, h), (255, 255, 0), 2)
    cv2.line(frame, (0, h // 2), (w, h // 2), (255, 255, 0), 2)
    label_positions = {
        "North": (10, 78), "South": (w // 2 + 10, 78),
        "East": (10, h // 2 + 30), "West": (w // 2 + 10, h // 2 + 30)
    }
    for lane, decision in decisions.items():
        if lane not in lanes or lane.startswith("_"):
            continue
        color = colors.get(decision["signal"], (0, 0, 255))
        pos = label_positions[lane]
        text = (f"{lane}: {decision['signal']} | vehicles={decision['vehicle_count']} "
                f"| waited={decision.get('waiting_time_seconds', 0):.0f}s")
        cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.48, color, 2)
        x1, y1, x2, y2 = lanes[lane]
        cv2.rectangle(frame, (x1 + 2, y1 + 2), (x2 - 2, y2 - 2), color, 3)

    return frame

def push_decision_to_esp32(decisions, backend_url, session_id=None):
    # Fire off the latest signal decision to the backend, which forwards
    # it to the physical ESP32 AND -- when session_id is set -- publishes
    # it onto that session's live SSE stream so the frontend updates in
    # near real time as each segment is decided, not just at the end.
    # Short timeout, never raises -- a failed push must not interrupt
    # video processing.
    if not backend_url:
        return {"status": "skipped", "reason": "no backend_url configured"}
    try:
        resp = requests.post(
            f"{backend_url}/esp32/sync",
            json={"signal_decisions": decisions, "session_id": session_id},
            timeout=5,
        )
        print(f"[ESP32 push] {resp.status_code} {resp.json() if resp.headers.get('content-type','').startswith('application/json') else resp.text}")
        return {"status": "sent", "http_status": resp.status_code}
    except Exception as e:
        print(f"[ESP32 push] failed (continuing processing): {e}")
        return {"status": "failed", "error": str(e)}

def process_traffic_video(video_url, backend_url=None, segment_seconds=None, session_id=None):
    t_start = time.time()
    backend_url = (backend_url or BACKEND_URL or "").rstrip("/")
    seg_secs = segment_seconds or SEGMENT_SECONDS
    controller = SignalController()

    temp_input      = f"/tmp/input_{uuid.uuid4()}.mp4"
    temp_output     = f"/tmp/output_{uuid.uuid4()}.mp4"
    temp_output_web = f"/tmp/output_web_{uuid.uuid4()}.mp4"

    print(f"1. Downloading video: {video_url}...")
    r = requests.get(video_url, stream=True, timeout=120)
    r.raise_for_status()
    with open(temp_input, 'wb') as f:
        for chunk in r.iter_content(65536):
            if chunk:
                f.write(chunk)

    cap = cv2.VideoCapture(temp_input)
    if not cap.isOpened():
        raise RuntimeError("Could not open downloaded video")

    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_vid_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0

    out_w, out_h = min(1280, orig_w), min(720, orig_h)
    if out_w % 2 != 0: out_w -= 1
    if out_h % 2 != 0: out_h -= 1

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_output, fourcc, fps, (out_w, out_h))

    process_every = max(1, fps // 4)
    frames_per_segment = max(1, int(round(seg_secs * fps)))

    frame_count = 0
    last_detections, last_decisions = [], {}

    total_lane_counts = {"North": 0, "South": 0, "East": 0, "West": 0}
    segment_lane_counts = {"North": 0, "South": 0, "East": 0, "West": 0}
    segment_samples = 0

    frame_data = []
    segment_decisions = []

    duration_sec = round(total_vid_frames / fps, 2) if total_vid_frames else None
    print(f"2. Processing FULL video ({total_vid_frames or '?'} frames, ~{duration_sec if duration_sec else '?'}s) "
          f"on device={device}... segment={seg_secs}s max_green={controller.max_green_seconds}s "
          f"max_wait={controller.max_wait_seconds}s | ESP32 push target: {backend_url or 'DISABLED'}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        if (orig_w, orig_h) != (out_w, out_h):
            frame = cv2.resize(frame, (out_w, out_h))

        lanes = divide_frame_into_lanes(frame)

        if frame_count % process_every == 0:
            results = model.predict(frame, device=device, conf=0.3, verbose=False)[0]
            last_detections = results.boxes.data.cpu().numpy() if results.boxes is not None else []
            lane_counts = {ln: count_vehicles_in_lane(last_detections, lb) for ln, lb in lanes.items()}

            for lane in total_lane_counts:
                total_lane_counts[lane] += lane_counts[lane]
                segment_lane_counts[lane] += lane_counts[lane]
            segment_samples += 1

            frame_data.append({
                "frame": frame_count,
                "time_sec": round(frame_count / fps, 1),
                "North": lane_counts["North"],
                "South": lane_counts["South"],
                "East":  lane_counts["East"],
                "West":  lane_counts["West"],
                "total": sum(lane_counts.values()),
            })

        for det in last_detections:
            if int(det[5]) in VEHICLE_CLASSES:
                x1, y1, x2, y2 = int(det[0]), int(det[1]), int(det[2]), int(det[3])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                cv2.putText(frame, VEHICLE_CLASSES[int(det[5])],
                            (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        if last_decisions:
            frame = draw_signal_overlay(frame, lanes, last_decisions)

        out.write(frame)

        if frame_count % frames_per_segment == 0:
            seg_decision = controller.decide(segment_lane_counts, seg_secs)
            last_decisions = seg_decision
            esp32_result = push_decision_to_esp32(seg_decision, backend_url, session_id)
            segment_decisions.append({
                "segment_index": len(segment_decisions),
                "start_sec": round((frame_count - frames_per_segment) / fps, 2),
                "end_sec": round(frame_count / fps, 2),
                "lane_counts": dict(segment_lane_counts),
                "decisions": seg_decision,
                "esp32_push": esp32_result,
            })
            ctrl = seg_decision.get("_controller", {})
            print(f"   → Segment {len(segment_decisions)} "
                  f"[{segment_decisions[-1]['start_sec']}s-{segment_decisions[-1]['end_sec']}s]: "
                  f"{segment_lane_counts} | GREEN={ctrl.get('selected_lane')} "
                  f"held={ctrl.get('green_elapsed_seconds')}s reason={ctrl.get('reason')} "
                  f"-> pushed to ESP32: {esp32_result['status']}")
            segment_lane_counts = {"North": 0, "South": 0, "East": 0, "West": 0}
            segment_samples = 0

    if segment_samples > 0:
        seg_decision = controller.decide(segment_lane_counts, seg_secs)
        last_decisions = seg_decision
        esp32_result = push_decision_to_esp32(seg_decision, backend_url, session_id)
        segment_decisions.append({
            "segment_index": len(segment_decisions),
            "start_sec": round((frame_count - (frame_count % frames_per_segment)) / fps, 2),
            "end_sec": round(frame_count / fps, 2),
            "lane_counts": dict(segment_lane_counts),
            "decisions": seg_decision,
            "esp32_push": esp32_result,
        })
        print(f"   → Final partial segment {len(segment_decisions)}: pushed to ESP32: {esp32_result['status']}")

    cap.release()
    out.release()

    print("3. Fast transcoding with ffmpeg...")
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -preset ultrafast -crf 30 -acodec aac {temp_output_web} -y -loglevel quiet")
    upload_file = temp_output_web if os.path.exists(temp_output_web) else temp_output

    print("4. Fast Cloudinary upload...")
    upload_result = cloudinary.uploader.upload(upload_file, resource_type="video", folder="traffic-ai/processed")
    processed_video_url = upload_result.get("secure_url", "")

    for f in [temp_input, temp_output, temp_output_web]:
        if os.path.exists(f):
            os.remove(f)

    # "Final" decision = the controller's live state at the end of
    # processing (NOT a recomputation from aggregate totals -- that
    # wouldn't make sense for a stateful controller that depends on
    # sequence/timing, not just cumulative counts).
    final_decisions = segment_decisions[-1]["decisions"] if segment_decisions else controller.decide(total_lane_counts, seg_secs)
    push_decision_to_esp32(final_decisions, backend_url, session_id)

    print(f"✓ Completed in {round(time.time() - t_start, 2)}s! ({len(segment_decisions)} segments, {frame_count} frames)")

    return {
        "lane_counts": total_lane_counts,
        "signal_decisions": final_decisions,
        "segment_decisions": segment_decisions,
        "frame_data": frame_data[::2] if len(frame_data) > 200 else frame_data,
        "processed_video_url": processed_video_url,
        "total_vehicles": sum(total_lane_counts.values()),
        "total_frames_processed": frame_count,
        "video_duration_seconds": duration_sec,
        "segment_seconds": seg_secs,
        "processing_time_seconds": round(time.time() - t_start, 2),
    }

def process_multi_lane_videos(video_urls, backend_url=None, segment_seconds=None, session_id=None):
    t_start = time.time()
    backend_url = (backend_url or BACKEND_URL or "").rstrip("/")
    seg_secs = segment_seconds or SEGMENT_SECONDS
    controller = SignalController()

    lanes = [l for l in ALL_LANES if video_urls.get(l)]
    if not lanes:
        raise ValueError("No lane videos provided. Expected a 'videos' dict with at least one of North/South/East/West.")

    temp_inputs, caps, fps_map, frames_per_segment, process_every = {}, {}, {}, {}, {}
    frames_per_first_segment = {}
    out_paths, writer_info = {}, {}

    print(f"1. Downloading {len(lanes)} lane video(s) in parallel: {lanes}")

    def _download_lane(lane, url):
        temp_in = f"/tmp/{lane.lower()}_in_{uuid.uuid4()}.mp4"
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        with open(temp_in, "wb") as f:
            for chunk in r.iter_content(65536):
                if chunk:
                    f.write(chunk)
        return lane, temp_in

    # Downloads are I/O-bound (waiting on Cloudinary), so a thread pool
    # gets a real wall-clock win here even under the GIL -- 4 lanes
    # downloading at once instead of one-after-another was previously
    # adding straight-up serial network latency before ANY processing
    # could even start.
    t_dl = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(lanes)) as pool:
        for lane, temp_in in pool.map(lambda args: _download_lane(*args),
                                       [(lane, video_urls[lane]) for lane in lanes]):
            temp_inputs[lane] = temp_in
    print(f"   ✓ All lane videos downloaded in {round(time.time() - t_dl, 2)}s")

    for lane in lanes:
        url = video_urls[lane]
        temp_in = temp_inputs[lane]

        cap = cv2.VideoCapture(temp_in)
        opened = cap.isOpened()

        fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_vid_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        duration_sec = round(total_vid_frames / fps, 2) if (fps and total_vid_frames) else None

        # Explicit per-lane diagnostics -- do NOT silently let "0 frames"
        # or "failed to open" turn into a quiet "0 vehicles" downstream.
        print(f"   [{lane}] video path: {temp_in}")
        print(f"   [{lane}] opened: {opened} | fps: {fps} | frames: {total_vid_frames} "
              f"| duration: {duration_sec}s | resolution: {orig_w}x{orig_h}")

        if not opened:
            raise RuntimeError(f"Could not open {lane} video (downloaded from {url})")
        if total_vid_frames == 0:
            print(f"   [{lane}] WARNING: frame count reported as 0 -- this lane may "
                  f"produce zero detections even though the file opened successfully.")
        caps[lane] = cap

        fps_map[lane] = fps
        frames_per_segment[lane] = max(1, int(round(seg_secs * fps)))
        frames_per_first_segment[lane] = max(1, int(round(FIRST_SEGMENT_SECONDS * fps)))
        process_every[lane] = max(1, fps // 4)

        out_w, out_h = min(1280, orig_w), min(720, orig_h)
        if out_w % 2: out_w -= 1
        if out_h % 2: out_h -= 1

        out_path = f"/tmp/{lane.lower()}_out_{uuid.uuid4()}.mp4"
        out_paths[lane] = out_path
        writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (out_w, out_h))
        writer_info[lane] = {"writer": writer, "out_w": out_w, "out_h": out_h, "orig_w": orig_w, "orig_h": orig_h}

    print(f"2. Processing videos for lanes {lanes} on device={device} "
          f"| first_segment={FIRST_SEGMENT_SECONDS}s (fast initial decision), "
          f"segment={seg_secs}s thereafter | max_green={controller.max_green_seconds}s "
          f"max_wait={controller.max_wait_seconds}s | ESP32 push target: {backend_url or 'DISABLED'}")

    total_lane_counts = {lane: 0 for lane in lanes}
    total_raw_detections = {lane: 0 for lane in lanes}
    segment_decisions = []
    frame_data = []
    finished = {lane: False for lane in lanes}
    segment_index = 0

    # ── INSTANT FIRST DECISION ──────────────────────────────────────────
    # Grab exactly ONE frame from each lane and run a SINGLE BATCHED YOLO
    # call across all of them together (one model.predict([...]) call, not
    # four separate ones) at a smaller inference size. This is the fastest
    # a real decision can possibly reach the frontend/ESP32 -- it exists
    # purely to kill "stuck on WAITING" latency. It's necessarily noisier
    # (one frame, smaller image) than a proper segment; the very next
    # segment (still short -- FIRST_SEGMENT_SECONDS) refines it with real
    # multi-frame sampling at full resolution.
    t_instant = time.time()
    instant_frames, instant_lane_order = [], []
    for lane in lanes:
        ret, frame = caps[lane].read()
        if not ret:
            finished[lane] = True
            continue
        info = writer_info[lane]
        if (info["orig_w"], info["orig_h"]) != (info["out_w"], info["out_h"]):
            frame = cv2.resize(frame, (info["out_w"], info["out_h"]))
        instant_frames.append(frame)
        instant_lane_order.append(lane)

    if instant_frames:
        batched_results = model.predict(
            instant_frames, device=device, conf=0.30, imgsz=FIRST_DECISION_IMGSZ, verbose=False
        )
        instant_counts = {lane: 0 for lane in lanes}
        instant_raw = {lane: 0 for lane in lanes}
        for lane, frame, result in zip(instant_lane_order, instant_frames, batched_results):
            dets = result.boxes.data.cpu().numpy() if result.boxes is not None else []
            instant_raw[lane] = len(dets)
            instant_counts[lane] = sum(1 for d in dets if int(d[5]) in VEHICLE_CLASSES)
            for d in dets:
                if int(d[5]) in VEHICLE_CLASSES:
                    x1, y1, x2, y2 = int(d[0]), int(d[1]), int(d[2]), int(d[3])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                    cv2.putText(frame, VEHICLE_CLASSES[int(d[5])], (x1, max(15, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
            writer_info[lane]["writer"].write(frame)
            total_lane_counts[lane] += instant_counts[lane]
            total_raw_detections[lane] += instant_raw[lane]

        full_instant = {l: instant_counts.get(l, 0) for l in ALL_LANES}
        full_instant_raw = {l: instant_raw.get(l, 0) for l in ALL_LANES}
        # segment_seconds=0.0: this is the controller's very FIRST call
        # (current_green is None), so it just picks the busiest lane --
        # it doesn't advance any green/waiting timers.
        instant_decision = controller.decide(full_instant, 0.0)
        esp32_result = push_decision_to_esp32(instant_decision, backend_url, session_id)
        segment_decisions.append({
            "segment_index": "instant",
            "segment_seconds": 0.0,
            "raw_yolo_detections": full_instant_raw,
            "lane_counts": full_instant,
            "decisions": instant_decision,
            "esp32_push": esp32_result,
        })
        ctrl = instant_decision.get("_controller", {})
        print(f"   → INSTANT decision in {round(time.time() - t_instant, 2)}s "
              f"(1 batched YOLO call, {len(instant_frames)} frames @ {FIRST_DECISION_IMGSZ}px): "
              f"raw_yolo={full_instant_raw} vehicles={full_instant} | "
              f"GREEN={ctrl.get('selected_lane')} reason={ctrl.get('reason')} "
              f"-> pushed to ESP32: {esp32_result['status']}")

    while not all(finished.values()):
        # The very first segment is intentionally short (FIRST_SEGMENT_SECONDS)
        # so a real decision reaches the frontend/ESP32 quickly instead of
        # making the UI wait out a full multi-lane SEGMENT_SECONDS pass on
        # (likely CPU) inference before anything appears. Every segment
        # after that uses the normal, steadier SEGMENT_SECONDS window.
        current_seg_secs = FIRST_SEGMENT_SECONDS if segment_index == 0 else seg_secs
        segment_lane_counts = {lane: 0 for lane in lanes}
        segment_raw_counts = {lane: 0 for lane in lanes}  # ALL YOLO detections, any class

        for lane in lanes:
            if finished[lane]:
                continue
            cap = caps[lane]
            info = writer_info[lane]
            frames_to_read = (
                frames_per_first_segment[lane] if segment_index == 0 else frames_per_segment[lane]
            )
            last_detections = []

            for i in range(frames_to_read):
                ret, frame = cap.read()
                if not ret:
                    finished[lane] = True
                    break

                if (info["orig_w"], info["orig_h"]) != (info["out_w"], info["out_h"]):
                    frame = cv2.resize(frame, (info["out_w"], info["out_h"]))

                if i % process_every[lane] == 0:
                    results = model.predict(frame, device=device, conf=0.30, verbose=False)[0]
                    last_detections = results.boxes.data.cpu().numpy() if results.boxes is not None else []
                    raw_count = len(last_detections)  # every class YOLO found, not just vehicles
                    vehicle_count = sum(1 for d in last_detections if int(d[5]) in VEHICLE_CLASSES)
                    segment_raw_counts[lane] += raw_count
                    total_raw_detections[lane] += raw_count
                    segment_lane_counts[lane] += vehicle_count
                    total_lane_counts[lane] += vehicle_count

                for d in last_detections:
                    if int(d[5]) in VEHICLE_CLASSES:
                        x1, y1, x2, y2 = int(d[0]), int(d[1]), int(d[2]), int(d[3])
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                        cv2.putText(frame, VEHICLE_CLASSES[int(d[5])],
                                    (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

                info["writer"].write(frame)

        full_counts = {l: segment_lane_counts.get(l, 0) for l in ALL_LANES}
        full_raw_counts = {l: segment_raw_counts.get(l, 0) for l in ALL_LANES}
        seg_decision = controller.decide(full_counts, current_seg_secs)
        esp32_result = push_decision_to_esp32(seg_decision, backend_url, session_id)
        segment_decisions.append({
            "segment_index": segment_index,
            "segment_seconds": current_seg_secs,
            "raw_yolo_detections": full_raw_counts,   # every class, before vehicle filtering / lane logic
            "lane_counts": full_counts,                # vehicle-class detections assigned to each lane's video
            "decisions": seg_decision,
            "esp32_push": esp32_result,
        })
        frame_data.append({
            "frame": segment_index,
            "time_sec": round(segment_index * seg_secs, 1),
            **full_counts,
            "total": sum(full_counts.values()),
        })
        ctrl = seg_decision.get("_controller", {})
        print(f"   → Segment {segment_index} ({current_seg_secs}s): "
              f"raw_yolo={full_raw_counts} vehicles={full_counts} | "
              f"GREEN={ctrl.get('selected_lane')} held={ctrl.get('green_elapsed_seconds')}s "
              f"reason={ctrl.get('reason')} -> pushed to ESP32: {esp32_result['status']}")
        if segment_index == 0 and sum(full_raw_counts.values()) > 0 and sum(full_counts.values()) == 0:
            print("   ⚠ YOLO is detecting objects but none are classified as vehicles "
                  "(car/motorcycle/bus/truck) in this segment -- check footage content/angle, "
                  "not the pipeline wiring.")
        elif segment_index == 0 and sum(full_raw_counts.values()) == 0:
            print("   ⚠ Zero raw YOLO detections in the first segment -- check the video diagnostics "
                  "above (opened/fps/frames) and confirm vehicles are actually visible/large enough "
                  "in this footage; this is a detection issue, not a lane-assignment issue.")
        segment_index += 1

    for lane in lanes:
        caps[lane].release()
        writer_info[lane]["writer"].release()

    print("3. Fast transcoding + uploading each lane's annotated video...")
    processed_video_urls = {}
    cleanup_paths = list(temp_inputs.values())
    for lane in lanes:
        out_path = out_paths[lane]
        web_path = out_path.replace(".mp4", "_web.mp4")
        os.system(f"ffmpeg -i {out_path} -vcodec libx264 -preset ultrafast -crf 30 -acodec aac {web_path} -y -loglevel quiet")
        upload_file = web_path if os.path.exists(web_path) else out_path
        upload_result = cloudinary.uploader.upload(
            upload_file, resource_type="video", folder=f"traffic-ai/processed/{lane.lower()}"
        )
        processed_video_urls[lane] = upload_result.get("secure_url", "")
        cleanup_paths += [out_path, web_path]

    for f in cleanup_paths:
        if os.path.exists(f):
            os.remove(f)

    full_total_counts = {l: total_lane_counts.get(l, 0) for l in ALL_LANES}
    final_decisions = segment_decisions[-1]["decisions"] if segment_decisions else controller.decide(full_total_counts, seg_secs)
    push_decision_to_esp32(final_decisions, backend_url, session_id)

    print(f"✓ Completed in {round(time.time() - t_start, 2)}s! ({len(segment_decisions)} segments)")

    return {
        "lane_counts": full_total_counts,
        "signal_decisions": final_decisions,
        "segment_decisions": segment_decisions,
        "frame_data": frame_data,
        "processed_video_url": next(iter(processed_video_urls.values()), ""),
        "processed_video_urls": processed_video_urls,
        "total_vehicles": sum(full_total_counts.values()),
        "segment_seconds": seg_secs,
        "lanes_processed": lanes,
        "processing_time_seconds": round(time.time() - t_start, 2),
    }

@app.get("/")
def root():
    return {"status": "Colab AI server running", "device": str(device), "backend_url": BACKEND_URL or None,
            "controller": {"min_green_seconds": MIN_GREEN_SECONDS, "max_green_seconds": MAX_GREEN_SECONDS,
                            "max_wait_seconds": MAX_WAIT_SECONDS, "clear_threshold_vehicles": CLEAR_THRESHOLD_VEHICLES,
                            "starvation_min_vehicles": STARVATION_MIN_VEHICLES, "starvation_max_vehicles": STARVATION_MAX_VEHICLES}}

@app.get("/health")
def health():
    return {"status": "healthy", "device": str(device), "model": "YOLOv8n"}

@app.post("/test-dynamic-decision")
def test_dynamic_decision(payload: dict):
    """
    Simulate the controller over multiple ticks WITHOUT needing a video.
    Body: {"lane_counts": {"North": 20, "South": 3, "East": 0, "West": 5},
           "ticks": 25, "segment_seconds": 5}
    Or, to vary counts over time:
    {"lane_counts_sequence": [{"North":20,...}, {"North":20,...}, ...], "segment_seconds": 5}
    Returns the full sequence of decisions so you can see rotation/clear-switch/starvation-relief happen.
    """
    seg_secs = payload.get("segment_seconds", SEGMENT_SECONDS)
    controller = SignalController(
        min_green_seconds=payload.get("min_green_seconds", MIN_GREEN_SECONDS),
        max_green_seconds=payload.get("max_green_seconds", MAX_GREEN_SECONDS),
        max_wait_seconds=payload.get("max_wait_seconds", MAX_WAIT_SECONDS),
        clear_threshold_vehicles=payload.get("clear_threshold_vehicles", CLEAR_THRESHOLD_VEHICLES),
        starvation_min=payload.get("starvation_min_vehicles", STARVATION_MIN_VEHICLES),
        starvation_max=payload.get("starvation_max_vehicles", STARVATION_MAX_VEHICLES),
    )
    if "lane_counts_sequence" in payload:
        sequence = payload["lane_counts_sequence"]
    else:
        counts = payload.get("lane_counts", {})
        ticks = int(payload.get("ticks", 20))
        sequence = [counts] * ticks

    results = []
    for i, counts in enumerate(sequence):
        decision = controller.decide(counts, seg_secs)
        results.append({"tick": i, "elapsed_seconds": round(i * seg_secs, 1), "lane_counts": counts, "decision": decision})
    return {"segment_seconds": seg_secs, "ticks": results}

@app.post("/process-video")
def process_video(req: VideoRequest):
    if req.videos:
        return process_multi_lane_videos(req.videos, req.backend_url, req.segment_seconds, req.session_id)
    if req.video_url:
        return process_traffic_video(req.video_url, req.backend_url, req.segment_seconds, req.session_id)
    raise HTTPException(status_code=400, detail="Provide either 'video_url' (legacy single-video mode) or 'videos' (per-lane mode).")
'''
with open("app.py", "w") as f:
    f.write(app_code)
print("✓ app.py created (greedy + clear-switch + rotation + starvation-relief controller)")


In [4]:
# CELL 4 — Start FastAPI on port 8001 and verify locally
import subprocess, time, requests

PORT = 8001

# Stop any previous server already bound to this port
!fuser -k {PORT}/tcp 2>/dev/null || true
time.sleep(1)

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(8)

if server.poll() is not None:
    print("❌ FastAPI failed to start")
    print("Return code:", server.returncode)
    print(server.stdout.read())
else:
    print(f"✓ FastAPI process is running on port {PORT}")
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10)
        print("Local FastAPI status:", response.status_code)
        print("Response:", response.json())
    except Exception as e:
        print("❌ FastAPI health check failed:", e)

✓ FastAPI process is running on port 8001
Local FastAPI status: 200
Response: {'status': 'healthy', 'device': 'cpu', 'model': 'YOLOv8n'}


In [ ]:
# CELL 5 — Clear stale ngrok sessions (fixes ERR_NGROK_108/ERR_NGROK_334:
# a previous Colab runtime crashed/disconnected without running CELL 8, so
# its tunnel is still "online" on ngrok's servers even though nothing local
# is running it anymore) and confirm the authtoken is actually set (fixes
# ERR_NGROK_4018).

from pyngrok import ngrok, conf
import os, time, requests

authtoken = os.environ.get("NGROK_AUTHTOKEN", "")
if not authtoken:
    raise RuntimeError("NGROK_AUTHTOKEN is empty — set it in CELL 2 before continuing.")

ngrok.set_auth_token(authtoken)

# 1. Kill the local agent/process (in case a previous run left one alive
#    in THIS runtime).
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception as e:
    print(f"No local tunnels to disconnect ({e})")

try:
    ngrok.kill()
except Exception as e:
    print(f"ngrok.kill() note: {e}")

!pkill -9 ngrok 2>/dev/null || true
time.sleep(2)

# 2. Stop remote sessions too -- these are what actually cause ERR_NGROK_108
# (3-session limit) and ERR_NGROK_334 (static domain already online), even
# when they were started from a DIFFERENT, now-dead Colab runtime that
# never got to run CELL 8. ngrok.kill() above can't touch these because
# they're not local processes.
# NOTE: this uses NGROK_API_KEY (from https://dashboard.ngrok.com/api-keys),
# which is a DIFFERENT credential from your authtoken.
api_key = os.environ.get("NGROK_API_KEY", "")
if not api_key or api_key.startswith("PASTE_YOUR_"):
    print("Skipping remote session cleanup (no NGROK_API_KEY set).")
    print("If CELL 6 fails with ERR_NGROK_108/334, set NGROK_API_KEY in CELL 2")
    print("or stop sessions manually at https://dashboard.ngrok.com/tunnels/agents")
else:
    headers = {"Authorization": f"Bearer {api_key}", "Ngrok-Version": "2", "Content-Type": "application/json"}
    try:
        resp = requests.get("https://api.ngrok.com/tunnel_sessions", headers=headers, timeout=15)
        resp.raise_for_status()
        remote_sessions = resp.json().get("tunnel_sessions", [])
        if not remote_sessions:
            print("No active remote tunnel sessions found on this account.")
        else:
            for sess in remote_sessions:
                sid = sess.get("id")
                print(f"Stopping stale remote session {sid} (started {sess.get('started_at')})...")
                try:
                    stop_resp = requests.post(
                        f"https://api.ngrok.com/tunnel_sessions/{sid}/stop",
                        headers=headers, json={}, timeout=15,
                    )
                    if stop_resp.status_code in (200, 202, 204):
                        print(f"  ✓ stopped {sid}")
                    else:
                        print(f"  ✗ could not stop {sid}: HTTP {stop_resp.status_code} {stop_resp.text[:200]}")
                except Exception as e:
                    print(f"  ✗ could not stop {sid}: {e}")
            time.sleep(3)  # give ngrok's backend a moment to release the domain
    except Exception as e:
        print(f"Could not query/stop remote sessions (non-fatal): {e}")
        print("If CELL 6 still fails, stop sessions manually at https://dashboard.ngrok.com/tunnels/agents")

print("✓ ngrok state reset")


In [7]:
# CELL 6 — Open the tunnel (exactly once) and verify both endpoints
from pyngrok import ngrok
import requests

tunnel = ngrok.connect(addr=PORT, proto="http", bind_tls=True)

print("=" * 60)
print("COLAB AI SERVER IS LIVE")
print("=" * 60)
print("COLAB API URL:", tunnel.public_url)
print("HEALTH:", tunnel.public_url + "/health")
print("DOCS:", tunnel.public_url + "/docs")
print("=" * 60)

# Local check
local = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10)
print("Local status:", local.status_code, local.json())

# Public check
public = requests.get(tunnel.public_url + "/health", timeout=20)
print("Public status:", public.status_code, public.json())

COLAB AI SERVER IS LIVE
COLAB API URL: https://YOUR_BACKEND_URL
HEALTH: https://YOUR_BACKEND_URL/health
DOCS: https://YOUR_BACKEND_URL/docs
Local status: 200 {'status': 'healthy', 'device': 'cpu', 'model': 'YOLOv8n'}
Public status: 200 {'status': 'healthy', 'device': 'cpu', 'model': 'YOLOv8n'}


In [ ]:
# CELL 7 — TEST: prove clear-switch, rotation, AND starvation relief all work
# Run this after CELL 6. It does not need a video or ESP32.
import requests

url = tunnel.public_url

def run_test(name, payload):
    resp = requests.post(f"{url}/test-dynamic-decision", json=payload, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    print(f"\n=== {name} ===")
    print(f"{'tick':>4} {'t(s)':>5}  {'green':>6}  {'reason':<20} counts")
    for t in data["ticks"]:
        ctrl = t["decision"]["_controller"]
        print(f"{t['tick']:>4} {t['elapsed_seconds']:>5}  {ctrl['selected_lane']:>6}  {ctrl['reason']:<20} {t['lane_counts']}")

# Test 1: North starts heavy then genuinely clears out -> should switch
# EARLY (before the 60s max-green timer) once North drops low and South
# has traffic waiting.
run_test("Early clear-switch", {
    "lane_counts_sequence": [{"North": 20, "South": 5, "East": 0, "West": 0}] * 3
                           + [{"North": 1,  "South": 5, "East": 0, "West": 0}] * 4,
    "segment_seconds": 5,
    "min_green_seconds": 15,
    "clear_threshold_vehicles": 2,
})

# Test 2: North/South both stay heavy and never clear, East stays light.
# Confirms max-green rotation AND starvation relief still work when the
# clear-switch rule never fires (nobody clears).
run_test("Rotation + starvation relief (nobody clears)", {
    "lane_counts": {"North": 20, "South": 18, "East": 3, "West": 0},
    "ticks": 20,
    "segment_seconds": 5,
    "max_green_seconds": 20,
    "max_wait_seconds": 35,
})

print()
print("✓ Test 1 should show 'traffic_cleared' switching South in early (before 60s).")
print("✓ Test 2 should show 'max_green_exceeded' rotating North<->South, and")
print("  'starvation_relief' eventually picking East despite fewest vehicles.")

## Cleanup

Run this when you're done so the tunnel doesn't linger and eat into your 3-session quota next time.

In [ ]:
# CELL 8 — Shut down the tunnel and server
from pyngrok import ngrok

try:
    ngrok.disconnect(tunnel.public_url)
except Exception:
    pass
ngrok.kill()
server.terminate()
print("✓ Tunnel and server stopped")